In [0]:
# Gold layer: dim_products (Kimball product dimension, SCD Type 2)
# Source: b2b_ecommerce.silver.products - silver is latest-state-only
#         (SCD Type 1 upserts, no history), so GOLD owns the change
#         history here, exactly like dim_companies: each run snapshots
#         silver's current state, diffs it against the last committed
#         snapshot and opens a new SCD2 version per product whenever a
#         tracked attribute actually changes.
# Why SCD2 (KPI-driven, requirements.md "Required KPIs"):
#         KPI 4 Gross Margin compares catalog price vs actual selling
#         price - the selling price is frozen per order line in the
#         fact, so historical margins are only correct against the
#         list_price AS OF the order date; KPI 3 Revenue by
#         Product/Category attributes revenue to the category as of
#         the sale (a re-categorized product must not drag its
#         historical revenue into the new category). Both attributes
#         change, so version history is required. KPIs 13/14
#         (supplier performance) only need product identity and join
#         facts to the dim as-of between __START_AT / __END_AT.
# Mechanism: AUTO CDC FROM SNAPSHOT into a Delta-backed streaming
#         table (same ingestion-time pattern as dim_companies -
#         silver is an upsert target with no change data feed but IS
#         a complete current-state snapshot). The view is read at
#         pipeline-run time and the run's ingestion time is the
#         snapshot version (collect-free - SDP forbids driver-side
#         collect() in pipeline source). Idempotent by construction:
#         an identical snapshot diffs to zero events, so retries and
#         no-op runs change nothing in the table.
# Grain:  one row per (product_id, __START_AT) version. Current row
#         = __END_AT IS NULL.
# Surrogate key: product_sk = crc32(product_id | updated_at of the
#         silver row that opened the version) - deterministic, stable
#         across refreshes, unique per version, assigned at insert and
#         never rewritten for unchanged products (user-directed
#         2026-11-18: crc32 convention - unsigned 32-bit, always
#         non-negative, SHORT max 10 digits; the "|" separator keeps
#         the two concatenated values unambiguous). Negative values
#         stay reserved per the Kimball default-member convention
#         (-1 Unknown, -2 Not Applicable, -3 Error/Invalid) - no seed
#         rows inserted. product_sk is EXCLUDED from history tracking:
#         a silver re-send that only bumps updated_at re-hashes the
#         key in place instead of opening a fake version.
# Columns: product_sk + product_id + sku + name + category +
#         list_price + is_active - the KPI/star-schema set (KPI 3
#         needs category, KPI 4 needs list_price; brand,
#         unit_of_measure, weight_kg, barcode, min_order_qty are used
#         by no KPI). created_at / updated_at dropped from the
#         snapshot view (gold timestamp policy: CDC plumbing and the
#         mixed ART/UTC source clocks never reach gold; updated_at
#         only feeds the surrogate-key hash). Validity = __START_AT /
#         __END_AT (the AUTO CDC convention); current row = __END_AT
#         IS NULL.
# Quality: no expectations - silver owns row-level validation, and
#         expectations are not allowed on AUTO CDC FROM SNAPSHOT
#         flows anyway; the SCD2 windows are maintained by the
#         AUTO CDC engine.
# Layout: liquid-clustered by product_sk (dims cluster by their
#         surrogate key - KPI views and facts join on it).
from pyspark import pipelines as dp
from pyspark.sql import functions as F

SILVER_PRODUCTS = "b2b_ecommerce.silver.products"


# Silver's complete current-state snapshot, plus the surrogate key.
# A batch temporary view: the engine reads it at each pipeline run
# (ingestion-time snapshot versioning). No driver-side actions anywhere.
# The select doubles as the column trim: brand, unit_of_measure,
# weight_kg, barcode, min_order_qty, created_at, updated_at stay out
# of gold (updated_at is consumed by the crc32 hash before the drop).
@dp.temporary_view(name="products_snapshot")
def products_snapshot():
    return (
        spark.read.table(SILVER_PRODUCTS)
        .select(
            # surrogate key: crc32 of the natural key + the silver row's own
            # updated_at (the change that opened this version) - unsigned,
            # non-negative, short (max 10 digits)
            F.crc32(
                F.concat_ws(
                    "|",
                    F.col("product_id").cast("string"),
                    F.col("updated_at").cast("string"),
                )
            ).alias("product_sk"),
            "product_id",
            "sku",
            "name",
            "category",
            "list_price",
            "is_active",
        )
    )


# Delta-backed streaming table: the AUTO CDC target, clustered on
# the surrogate key (dims cluster by their surrogate key).
dp.create_streaming_table(
    name="dim_products",
    comment=(
        "Kimball product dimension, SCD Type 2: one row per (product_id, __START_AT) "
        "version, built from the b2b_ecommerce.silver.products current-state snapshot "
        "via AUTO CDC FROM SNAPSHOT (ingestion-time pattern) keyed on product_id. SCD2 "
        "is KPI-driven: KPI 4 (gross margin) needs list_price as of the order date and "
        "KPI 3 (revenue by category) needs category as of the sale; a new version opens "
        "only when a tracked attribute changes (product_sk itself never opens one); "
        "identical snapshots diff to zero events, so runs are idempotent. Surrogate key "
        "product_sk = crc32(product_id | updated_at) - unsigned 32-bit, non-negative, "
        "short. created_at/updated_at and non-KPI descriptive columns excluded per the "
        "gold timestamp policy. Current row: __END_AT IS NULL."
    ),
    cluster_by=["product_sk"],
)


# The auto-merge load: diff each snapshot against the last committed
# one and apply inserts/updates as SCD2 versions.
dp.create_auto_cdc_from_snapshot_flow(
    target="dim_products",
    source="products_snapshot",  # ingestion-time snapshot versioning
    keys=["product_id"],
    stored_as_scd_type=2,
    # product_sk alone must not open versions (updated_at-only re-sends);
    # every business attribute is tracked
    track_history_except_column_list=["product_sk"],
)